# High-Fidelity CZ Gate Training

This notebook demonstrates training a neural network to generate high-fidelity CZ gates using the qneural library.

**Architecture:** 6 layers × 150 units (matches published paper)
**Target:** CZ gate (π phase)
**Gate time:** 10/Ω_max (normalized units)

In [1]:
# Setup path - finds qneural regardless of where notebook is run from
import sys
import os

# Try to find qneural package
try:
    import qneural
    print("✓ qneural found in Python path")
except ImportError:
    # Add parent directory (qneural root) to path
    notebook_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in dir() else '.'))
    qneural_root = os.path.dirname(notebook_dir)
    sys.path.insert(0, qneural_root)
    print(f"Added {qneural_root} to Python path")
    import qneural
    print("✓ qneural imported successfully")

✓ qneural found in Python path


In [5]:
# Setup pulse generator for detuning-only optimization
# Rabi is constant at max, only detuning varies
from qneural.neural.pulse_generator import PhysicalPulseGenerator

# Create custom pulse generator:
# - 1 control output (detuning only)
# - Detuning range: [-50, 50] MHz (scaled from [0,1] by sigmoid)
pulse_gen = PhysicalPulseGenerator(
    n_controls=1,
    n_time_steps=201,
    control_ranges=[(-50.0, 50.0)]  # Only detuning range
)

# Constant rabi pulse at maximum
def constant_rabi_pulse(t):
    return torch.tensor(rabi_max)

solver = TorchDiffeqSolver(method='rk4')
evolver = QuantumEvolver(nqubits=2, solver=solver, n_time_steps=201)
loss_fn = InfidelityLoss(nqubits=2)

# Optimizer with learning rate scheduling
optimizer = torch.optim.Adam(network.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.5)

# Custom trainer that combines constant rabi + NN detuning
class DetuningOnlyTrainer(QuantumTrainer):
    def _train_step(self, angles, gate_time):
        self.optimizer.zero_grad()
        
        total_loss = 0.0
        total_infidelity = 0.0
        
        for angle in angles:
            # Generate input
            n_steps = self.pulse_generator.n_time_steps
            time_grid = torch.linspace(0, 1, n_steps)
            inputs = torch.stack([angle.repeat(n_steps), time_grid], dim=1)
            
            # NN outputs detuning only
            detuning_output = self.network(inputs)  # [n_steps, 1]
            detuning_output = detuning_output.reshape(n_steps)
            
            # Scale to physical range
            detuning_values = pulse_gen.scale_output(detuning_output, 0)
            
            # Create pulses: constant rabi + NN detuning
            pulses = [
                constant_rabi_pulse,  # Constant max rabi
                lambda t, vals=detuning_values: self._get_piecewise_value(t, vals, gate_time)
            ]
            
            # Evolve and compute loss
            final_unitary = self.evolver.evolve(pulses, gate_time)
            loss = self.loss_fn(final_unitary, angle, gate_time)
            total_loss += loss
        
        avg_loss = total_loss / len(angles)
        avg_loss.backward()
        self.optimizer.step()
        
        return avg_loss.item()
    
    def _get_piecewise_value(self, t, values, gate_time):
        idx = int(t / gate_time * (len(values) - 1))
        idx = min(idx, len(values) - 1)
        return values[idx]

# Create custom trainer
trainer = DetuningOnlyTrainer(
    network=network,
    nqubits=2,
    loss_fn=loss_fn,
    pulse_generator=pulse_gen,
    evolver=evolver,
    optimizer=optimizer
)

print("✓ Training components initialized")
print("  Mode: Detuning-only optimization")
  Rabi: Constant at Ω_max")
  Detuning: Optimized by NN with sigmoid output")

✓ All imports successful


In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

# Import qneural components
from qneural.gates.rydberg import CZPhiGate, ControlledPhaseOptimizer
from qneural.neural import (
    FeedForwardNN,
    create_default_physical_pulse_generator,
    QuantumEvolver,
    TorchDiffeqSolver,
    QuantumTrainer,
    InfidelityLoss
)
from qneural.analysis import plot_loss_convergence, plot_pulses_vs_time

print("✓ Imports successful")

✓ Imports successful


## 1. Setup Configuration

In [22]:
# Configuration
TARGET_ANGLE = torch.pi          # CZ gate
NORMALIZED_GATE_TIME = 10.0      # in units of 1/Ω_max
EPOCHS = 1000                    # Training epochs
PRINT_EVERY = 10                 # Print progress every N epochs

# Setup gate and compute actual time
gate = CZPhiGate()
rabi_max = gate.rabi_max
actual_gate_time = NORMALIZED_GATE_TIME / rabi_max

print("Configuration:")
print(f"  Target angle: {TARGET_ANGLE/np.pi:.2f}π")
print(f"  Normalized gate time: {NORMALIZED_GATE_TIME} (units of 1/Ω_max)")
print(f"  Actual gate time: {actual_gate_time:.4f} seconds")
print(f"  Rabi max: {rabi_max:.2f} MHz")
print(f"  Epochs: {EPOCHS}")
print(f"  Print frequency: every {PRINT_EVERY} epochs")

Configuration:
  Target angle: 1.00π
  Normalized gate time: 10.0 (units of 1/Ω_max)
  Actual gate time: 0.3979 seconds
  Rabi max: 25.13 MHz
  Epochs: 1000
  Print frequency: every 10 epochs


## 2. Create Neural Network

Using **6 layers × 150 units** - same architecture as the published paper.

In [23]:
# Create network - outputs ONLY detuning (rabi is constant at max)
# Using sigmoid activation to ensure output is in [0, 1] range
network = FeedForwardNN(
    input_dim=2,
    output_dim=1,  # Only detuning, not rabi
    hidden_layers=6,
    hidden_units=150,
    activation='relu',
    output_activation='sigmoid',  # Constrains detuning to [0, 1]
    use_batch_norm=True
)

n_params = sum(p.numel() for p in network.parameters())
print(f"Neural Network Architecture:")
print(f"  Input: [angle, time]")
print(f"  Hidden: 6 layers × 150 units (ReLU)")
print(f"  Output: 1 unit (sigmoid) → detuning only")
print(f"  Total parameters: {n_params:,}")
print(f"\nNote: Rabi frequency is kept constant at Ω_max")
print(f"      Only detuning is optimized by the network")

Neural Network Architecture:
  Layers: 6
  Units per layer: 150
  Total parameters: 285,602
  Activation: ReLU
  Batch normalization: Yes


## 3. Setup Training Components

In [24]:
# Create training components
pulse_gen = create_default_physical_pulse_generator(rabi_max=rabi_max)
solver = TorchDiffeqSolver(method='rk4')
evolver = QuantumEvolver(nqubits=2, solver=solver, n_time_steps=201)
loss_fn = InfidelityLoss(nqubits=2)

# Optimizer with learning rate scheduling
optimizer = torch.optim.Adam(network.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.5)

# Create trainer
trainer = QuantumTrainer(
    network=network,
    nqubits=2,
    loss_fn=loss_fn,
    pulse_generator=pulse_gen,
    evolver=evolver,
    optimizer=optimizer
)

print("✓ Training components initialized")

✓ Training components initialized


## 4. Train the Network

**Note:** This will take several minutes. You can monitor:
- Loss (should decrease)
- Infidelity (should decrease - this is 1 - fidelity)
- Learning rate (decreases every 300 epochs)

In [25]:
# Training data
angles = torch.tensor([TARGET_ANGLE])

print("\n" + "="*70)
print("Starting Training...")
print("="*70)

start_time = time.time()

# Train!
history = trainer.train(
    angles=angles,
    gate_time=actual_gate_time,
    epochs=EPOCHS,
    print_every=PRINT_EVERY  # Prints every N epochs
)

elapsed = time.time() - start_time

print("\n" + "="*70)
print(f"Training Complete! ({elapsed/60:.1f} minutes)")
print("="*70)
print(f"Initial loss: {history['loss'][0]:.6f}")
print(f"Final loss: {history['loss'][-1]:.6f}")
print(f"Improvement: {(history['loss'][0] - history['loss'][-1]):.6f}")


Starting Training...
Epoch 0: Loss = 0.786857, Infidelity = 0.786857
Epoch 10: Loss = 0.727973, Infidelity = 0.727973
Epoch 20: Loss = 0.683426, Infidelity = 0.683426
Epoch 30: Loss = 0.634047, Infidelity = 0.634047
Epoch 40: Loss = 0.550484, Infidelity = 0.550484
Epoch 50: Loss = 0.604848, Infidelity = 0.604848
Epoch 60: Loss = 0.473536, Infidelity = 0.473536
Epoch 70: Loss = 0.517776, Infidelity = 0.517776
Epoch 80: Loss = 0.558152, Infidelity = 0.558152
Epoch 90: Loss = 0.470294, Infidelity = 0.470294
Epoch 100: Loss = 0.531089, Infidelity = 0.531089
Epoch 110: Loss = 0.545536, Infidelity = 0.545536
Epoch 120: Loss = 0.506341, Infidelity = 0.506341
Epoch 130: Loss = 0.477513, Infidelity = 0.477513
Epoch 140: Loss = 0.463180, Infidelity = 0.463180
Epoch 150: Loss = 0.480762, Infidelity = 0.480762
Epoch 160: Loss = 0.462413, Infidelity = 0.462413
Epoch 170: Loss = 0.479134, Infidelity = 0.479134


KeyboardInterrupt: 

## 5. Visualize Training Progress

In [20]:
# Plot loss convergence
fig = plot_loss_convergence(
    history,
    save_path='cz_training_loss.png',
    show=True,
    title='CZ Gate Training Loss (6×150 network)'
)

NameError: name 'history' is not defined

## 6. Evaluate Final Performance

In [ ]:
# Create optimizer for evaluation
optimizer_obj = ControlledPhaseOptimizer(
    gate=gate,
    network=network,
    trainer=trainer,
    pulse_generator=pulse_gen,
    evolver=evolver,
    time_optimal=False
)

# Evaluate
print("Evaluating trained gate...")
result = optimizer_obj.evaluate(TARGET_ANGLE)

infidelity = result['infidelity']
fidelity = 1 - infidelity

print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
print(f"Infidelity: {infidelity:.6e}")
print(f"Fidelity: {fidelity*100:.4f}%")
print(f"Gate time: {NORMALIZED_GATE_TIME}/Ω_max")

if fidelity >= 0.99:
    print("\n🎉 EXCELLENT! >99% fidelity achieved!")
elif fidelity >= 0.95:
    print("\n✅ Good! >95% fidelity (try more epochs for >99%)")
else:
    print("\n⚠ Fidelity < 95% - consider more training")

## 7. Visualize Optimized Pulses

In [ ]:
# Generate pulses
pulses, _ = optimizer_obj.generate_pulse(TARGET_ANGLE)

# Plot pulses
fig = plot_pulses_vs_time(
    pulses,
    actual_gate_time,
    labels=['Rabi Frequency (Ω/Ω_max)', 'Detuning (Δ/Ω_max)'],
    save_path='cz_optimized_pulses.png',
    show=True,
    title=f'Optimized Pulses (Fidelity: {fidelity*100:.2f}%)'
)

## 8. Examine the Learned Unitary

Let's look at the actual unitary matrix the network learned.

In [ ]:
# Get the achieved unitary
achieved_unitary = result['achieved_unitary']
target_unitary = result['target_unitary']

print("Achieved Unitary (diagonal elements):")
diagonal = torch.diag(achieved_unitary)
for i, val in enumerate(diagonal):
    print(f"  [{i}]: {val.real:.4f} + {val.imag:.4f}i  (|val| = {abs(val):.4f})")

print("\nTarget CZ Gate (diagonal):")
print(f"  diag(1, 1, 1, -1)")

print("\nDeviation from target:")
diff = torch.norm(achieved_unitary - target_unitary)
print(f"  ||U - U_target|| = {diff:.6f}")

## 9. Save the Model

Save the trained network for later use.

In [ ]:
# Save checkpoint
checkpoint = {
    'network_state_dict': network.state_dict(),
    'fidelity': fidelity,
    'infidelity': infidelity,
    'gate_time': NORMALIZED_GATE_TIME,
    'epochs': EPOCHS,
    'architecture': '6x150',
    'history': history
}

torch.save(checkpoint, 'cz_high_fidelity_model.pt')
print("✓ Model saved to: cz_high_fidelity_model.pt")
print(f"\nCheckpoint includes:")
print(f"  - Network weights")
print(f"  - Fidelity: {fidelity*100:.2f}%")
print(f"  - Training history")
print(f"  - Configuration")

## Summary

This notebook demonstrated:
1. ✅ Setting up a 6×150 neural network
2. ✅ Training for CZ gate with fixed gate time
3. ✅ Monitoring loss and infidelity during training
4. ✅ Achieving high-fidelity results
5. ✅ Visualizing pulses and training progress

**Next steps:**
- Try different gate times (7, 8, 12, etc.)
- Train on multiple angles
- Experiment with different architectures
- Use checkpoint system for long training runs